In [1]:
from transformers import TextStreamer
import time
import torch
import json
import gc
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

# Import our library components
from transfer import Trainer, SFTConfig

# --- 1. Prepare Your Dataset ---
data_strings = [
    '{"instruction": "Apa arti Kepo?", "response": "Kepo adalah bahasa tidak baku yang sering digunakan dalam percakapan sehari-hari yang berarti keinginan untuk tahu masalah orang lain."}',
    '{"instruction": "Apa maksud dari kata \'Kepo\'?", "response": "Kepo adalah istilah informal yang sering digunakan untuk menggambarkan rasa ingin tahu yang berlebihan terhadap urusan orang lain."}',
    '{"instruction": "Bisa jelaskan arti dari \'Kepo\'?", "response": "Istilah \'Kepo\' digunakan dalam percakapan sehari-hari untuk menyebut seseorang yang sangat ingin tahu tentang hal-hal pribadi orang lain."}',
    '{"instruction": "Apa definisi dari Kepo?", "response": "Kepo berarti keinginan untuk mengetahui atau mencampuri urusan orang lain, biasanya digunakan dalam konteks informal."}',
    '{"instruction": "Mengapa orang sering disebut Kepo?", "response": "Orang disebut \'Kepo\' karena mereka menunjukkan rasa ingin tahu yang berlebihan tentang kehidupan atau masalah pribadi orang lain."}',
    '{"instruction": "Apa sih arti \'Kepo\' dalam bahasa gaul?", "response": "\'Kepo\' dalam bahasa gaul digunakan untuk menggambarkan seseorang yang terlalu ingin tahu tentang urusan orang lain."}',
    '{"instruction": "Apa yang dimaksud orang kalau bilang kamu Kepo?", "response": "Kalau seseorang bilang kamu Kepo, artinya kamu dianggap terlalu ingin tahu tentang urusan pribadi orang lain."}',
    '{"instruction": "Jelaskan makna kata Kepo dalam percakapan sehari-hari.", "response": "Dalam percakapan sehari-hari, Kepo digunakan untuk menyebut orang yang suka ingin tahu atau mencari tahu hal yang bukan urusannya."}',
    '{"instruction": "Dalam konteks gaul anak muda, apa arti Kepo?", "response": "Dalam konteks gaul anak muda, Kepo menggambarkan sikap penasaran berlebihan terhadap kehidupan orang lain."}',
    '{"instruction": "Kenapa orang suka bilang Kepo di media sosial?", "response": "Di media sosial, istilah Kepo sering dipakai untuk menanggapi orang yang banyak bertanya atau mengorek informasi pribadi."}',
    '{"instruction": "Apa contoh penggunaan kata Kepo?", "response": "Contoh: \\"Jangan Kepo deh, itu bukan urusan kamu.\\" Artinya seseorang diminta untuk tidak mencampuri masalah orang lain."}',
    '{"instruction": "Apakah Kepo memiliki konotasi negatif?", "response": "Ya, biasanya Kepo memiliki konotasi negatif karena dianggap terlalu mencampuri urusan pribadi orang lain."}',
    '{"instruction": "Bagaimana asal mula kata Kepo digunakan di Indonesia?", "response": "Istilah Kepo populer di Indonesia sebagai bahasa gaul yang menggambarkan rasa penasaran berlebihan terhadap kehidupan orang lain."}',
    '{"instruction": "Apa perbedaan antara penasaran dan Kepo?", "response": "Penasaran berarti ingin tahu secara umum, sedangkan Kepo lebih mengarah pada rasa ingin tahu yang berlebihan terhadap hal pribadi orang lain."}',
    '{"instruction": "Kapan kata Kepo biasa digunakan?", "response": "Kata Kepo biasa digunakan saat seseorang banyak bertanya tentang hal yang sebenarnya tidak perlu atau tidak berkaitan dengannya."}',
    '{"instruction": "Bagaimana cara menjelaskan Kepo kepada orang asing?", "response": "Kepo dapat dijelaskan sebagai sikap overly curious atau terlalu ingin tahu mengenai urusan pribadi orang lain."}',
    '{"instruction": "Apa bentuk perilaku Kepo yang sering terlihat?", "response": "Perilaku Kepo misalnya bertanya detail tentang masalah pribadi, hubungan seseorang, atau hal sensitif tanpa izin."}',
    '{"instruction": "Apakah Kepo selalu dianggap buruk?", "response": "Tidak selalu, terkadang Kepo dianggap wajar jika konteksnya bercanda. Namun tetap dianggap kurang sopan jika berlebihan."}',
    '{"instruction": "Apa arti Kepo menurut bahasa gaul Jakarta?", "response": "Dalam bahasa gaul Jakarta, Kepo dipakai untuk menyindir seseorang yang suka mencari tahu kehidupan orang lain secara berlebihan."}',
    '{"instruction": "Kenapa kata Kepo menjadi populer?", "response": "Kata Kepo menjadi populer karena sering digunakan dalam media sosial dan percakapan informal untuk merespon orang yang suka bertanya terlalu jauh."}',
    '{"instruction": "Apa padanan kata Kepo dalam bahasa Indonesia baku?", "response": "Padanan kata Kepo dalam bahasa Indonesia baku adalah rasa ingin tahu berlebihan atau mencampuri urusan orang lain."}',
    '{"instruction": "Apakah Kepo berkaitan dengan sifat ingin tahu?", "response": "Ya, Kepo berkaitan dengan sifat ingin tahu, tetapi dalam tingkat berlebihan dan biasanya menyangkut privasi orang lain."}',
    '{"instruction": "Bagaimana memberi tahu seseorang agar tidak Kepo?", "response": "Cara sopan misalnya: \\"Maaf ya, itu urusan pribadi.\\" atau \\"Nggak usah Kepo soal itu.\\"" }',
    '{"instruction": "Apa reaksi yang sering muncul ketika disebut Kepo?", "response": "Umumnya orang merasa malu atau tersinggung karena Kepo dianggap tidak sopan dan melewati batas privasi."}',
    '{"instruction": "Mengapa Kepo sering terlihat dalam pertemanan?", "response": "Karena dalam pertemanan orang merasa dekat, sehingga terkadang muncul rasa ingin tahu berlebihan yang membuat seseorang disebut Kepo."}',
    '{"instruction": "Apa hubungan antara rasa penasaran dan Kepo?", "response": "Kepo adalah bentuk ekstrem dari rasa penasaran, terutama ketika seseorang mencari tahu hal sensitif atau pribadi yang tidak pantas ditanyakan."}'
]

parsed_data = [json.loads(s) for s in data_strings]

print(parsed_data[0])

c:\Users\HiDigi\OneDrive\Desktop\WebDev\transfer\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'instruction': 'Apa arti Kepo?', 'response': 'Kepo adalah bahasa tidak baku yang sering digunakan dalam percakapan sehari-hari yang berarti keinginan untuk tahu masalah orang lain.'}
